# Regularized Logistic Regression Models

This notebook implements and evaluates logistic regression with L1 (Lasso) and L2 (Ridge) regularization.

**Goal**: Manage variance and improve generalization through regularization.

## Models:
1. **Logistic Regression with L1 (Lasso)**: Performs feature selection by driving some coefficients to zero
2. **Logistic Regression with L2 (Ridge)**: Shrinks coefficients but keeps all features

## Evaluation Metrics:
- **F1-Score** (primary): Mean ± standard deviation across 5 folds
- **ROC-AUC**: Discriminative ability
- **Precision** and **Recall**

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Import the evaluate_model function
from evaluate_model import evaluate_model

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load preprocessed training data
print("Loading preprocessed training data...")
train_df = pd.read_csv('../data/train_data_preprocessed.csv')
print(f"Training data shape: {train_df.shape}")

# Prepare features and target (data is already preprocessed)
X = train_df.drop('Revenue', axis=1)
y = train_df['Revenue']

print(f"\nFeatures shape: {X.shape}")
print(f"Features: {list(X.columns)}")

## 2. Feature Standardization

**Important**: Standardization is fit only on training folds within CV to prevent leakage.

In [ ]:
# Setup cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# We'll standardize features within the pipeline to prevent leakage
from sklearn.pipeline import Pipeline

print("Standardization will be applied within CV folds to prevent data leakage.")

## 3. L1 Regularization (Lasso)

Lasso performs feature selection by driving some coefficients to exactly zero.

In [ ]:
# Create pipeline with standardization and L1 logistic regression
pipeline_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(penalty='l1', solver='saga', max_iter=1000, random_state=42))
])

# Hyperparameter grid for L1
param_grid_l1 = {
    'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# Grid search with cross-validation
grid_search_l1 = GridSearchCV(
    pipeline_l1, param_grid_l1, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Training L1 Logistic Regression with Grid Search...")
grid_search_l1.fit(X, y)

print(f"\nBest parameters: {grid_search_l1.best_params_}")
print(f"Best F1-Score: {grid_search_l1.best_score_:.4f}")

In [ ]:
# Evaluate best L1 model
results_l1 = evaluate_model(grid_search_l1.best_estimator_, X, y, cv, "Logistic Regression (L1 - Lasso)")

## 4. L2 Regularization (Ridge)

Ridge regularization shrinks coefficients but keeps all features.

In [ ]:
# Create pipeline with standardization and L2 logistic regression
pipeline_l2 = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=42))
])

# Hyperparameter grid for L2
param_grid_l2 = {
    'classifier__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

# Grid search with cross-validation
grid_search_l2 = GridSearchCV(
    pipeline_l2, param_grid_l2, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)

print("Training L2 Logistic Regression with Grid Search...")
grid_search_l2.fit(X, y)

print(f"\nBest parameters: {grid_search_l2.best_params_}")
print(f"Best F1-Score: {grid_search_l2.best_score_:.4f}")

In [ ]:
# Evaluate best L2 model
results_l2 = evaluate_model(grid_search_l2.best_estimator_, X, y, cv, "Logistic Regression (L2 - Ridge)")

## 5. Model Comparison

In [ ]:
print(f"\n{'='*60}")
print("REGULARIZED LOGISTIC REGRESSION COMPARISON")
print(f"{'='*60}")
print(f"\n{'Metric':<15} {'L1 (Lasso)':<15} {'L2 (Ridge)':<15}")
print(f"{'-'*45}")
print(f"{'F1-Score':<15} {results_l1['cv_f1']:<15.4f} {results_l2['cv_f1']:<15.4f}")
print(f"{'Precision':<15} {results_l1['cv_precision']:<15.4f} {results_l2['cv_precision']:<15.4f}")
print(f"{'Recall':<15} {results_l1['cv_recall']:<15.4f} {results_l2['cv_recall']:<15.4f}")
print(f"{'ROC-AUC':<15} {results_l1['cv_roc_auc']:<15.4f} {results_l2['cv_roc_auc']:<15.4f}")

print(f"\n{'='*60}")
print("KEY INSIGHTS")
print(f"{'='*60}")
print(f"\n1. L1 (Lasso):")
print(f"   - Best C: {grid_search_l1.best_params_['classifier__C']}")
print(f"   - Performs feature selection")
print(f"\n2. L2 (Ridge):")
print(f"   - Best C: {grid_search_l2.best_params_['classifier__C']}")
print(f"   - Shrinks all coefficients")

## 6. Summary

**Key Principles Applied:**
- Standardization fit only on training folds (via Pipeline) to prevent leakage
- 5-fold stratified cross-validation for robust performance estimates
- Hyperparameter tuning via GridSearchCV
- Performance reported as mean ± std across folds

**Next Steps:**
- Compare with tree-based models
- Explore nonlinear feature interactions